# <center style="font-weight: bold; color: #0098cd;">Procesamiento del lenguaje natural: clasificación y estructuración de datos</center>

## 1. Introducción

### 1.1 Objetivo
transformar texto en datos estructurados útiles para análisis y modelado

👉 siguiente paso: definir **estructura exacta del JSON final + diccionario de normalización LATAM**
Ahí es donde tu TFM pasa de bueno a muy sólido.

### 1.2 Contexto dentro del sistema completo

### 1.3 Requisitos de esta fase

  * clasificación de intención
  * extracción de entidades (NER)
  * normalización

## 2. Preparación del entorno de trabajo

En esta sección se define el entorno técnico necesario para el desarrollo del pipeline de preprocesamiento de audio. Se incluyen la instalación de dependencias, la importación de librerías especializadas y la configuración de rutas de trabajo.

El objetivo es garantizar un entorno reproducible y estructurado que permita ejecutar el flujo completo de procesamiento sobre diferentes conjuntos de datos sin modificaciones en la lógica del código. Asimismo, se establece una organización clara de los directorios que facilita la separación entre datos originales, datos intermedios y resultados finales.

### 2.1 Instalación de dependencias

En este apartado se especifica la instalación de las librerías necesarias para la ejecución del *notebook*. Todas las dependencias se gestionan mediante un archivo `requirements.txt`, lo que permite replicar el entorno de ejecución de forma controlada y consistente.

#### 2.1.1 Configuración del entorno

Este notebook requiere la instalación previa de las dependencias del proyecto.

Ejecutar en terminal:

```bash
pip install -r requirements.txt
python -m spacy download es_core_news_md

### 2.2 Importación de librerías

Se importan las librerías necesarias para las distintas fases del procesamiento, incluyendo manipulación de señal, tratamiento de audio, modelos de detección de voz, análisis de datos y visualización.

Se organiza por bloques funcionales para permitir identificar claramente el propósito de cada conjunto de herramientas dentro del *pipeline*, facilitando la mantenibilidad y comprensión del código.

In [ ]:
# ==============================
# Gestión de rutas y sistema
# ==============================
from pathlib import Path                  # gestión de rutas del sistema de archivos
import json                               # lectura y escritura de ficheros JSON

# ==============================
# Manipulación de datos
# ==============================
import pandas as pd                       # manejo de datos tabulares
import numpy as np                        # operaciones numéricas

# ==============================
# Procesamiento de texto
# ==============================
import re                                 # expresiones regulares
import unicodedata                        # normalización unicode
from unidecode import unidecode           # eliminación de acentos y normalización simplificada

# ==============================
# NLP - modelos y pipelines
# ==============================
import spacy                                           # procesamiento lingüístico y NER
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # modelos tipo BERT (BETO)
from sklearn.preprocessing import LabelEncoder         # codificación de etiquetas para clasificación
from sklearn.model_selection import train_test_split   # partición de datos
from sklearn.linear_model import LogisticRegression    # modelo baseline de clasificación
from sklearn.feature_extraction.text import TfidfVectorizer  # vectorización de texto con TF-IDF
from sklearn.metrics import classification_report, confusion_matrix  # evaluación de modelos

# ==============================
# Normalización y matching
# ==============================
from rapidfuzz import fuzz, process       # comparación difusa de texto (matching de términos)

# ==============================
# Utilidades y control
# ==============================
from tqdm import tqdm                     # barra de progreso
import joblib                             # guardado y carga de modelos


import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import torch
from transformers import Trainer, TrainingArguments, logging
import joblib
import random
from spacy.training.example import Example
from spacy.training import offsets_to_biluo_tags
from spacy.scorer import Scorer
import re



### 2.3 Gestión y configuración de rutas

In [ ]:
# =========================
# DETECCIÓN DE RAÍZ
# =========================

project_root = Path.cwd()

while not (project_root / "data").exists():
    project_root = project_root.parent


# =========================
# DIRECTORIOS
# =========================

# Ruta raíz de datos
data_dir = project_root / "data"


# =========================
# CONFIGURACIÓN
# =========================

# Carpeta de configuración
config_dir = data_dir / "config"

# Subcarpetas de configuración
domain_config_dir = config_dir / "domain"
ner_config_dir = config_dir / "ner"
asr_config_dir = config_dir / "asr"

# Archivos de configuración
domain_terms_path = domain_config_dir / "coffee_cacao_terms.json"
ner_entities_path = ner_config_dir / "ner_entities.json"


# =========================
# DATASETS
# =========================

datasets_dir = data_dir / "datasets"
ner_dataset_dir = datasets_dir / "ner"

# Ruta correcta
ner_dataset_path = ner_dataset_dir / "ner_dataset.json"


# =========================
# TRANSCRIPCIONES (INPUT NLP)
# =========================

transcriptions_dir = data_dir / "transcriptions"

pred_asr_dir = transcriptions_dir / "asr_output"

ground_truth_dir = transcriptions_dir / "ground_truth"


# =========================
# SALIDA NLP
# =========================

structured_data_dir = data_dir / "structured_data"

classification_dir = structured_data_dir / "classification"
ner_dir = structured_data_dir / "ner"
final_output_dir = structured_data_dir / "final_output"


# =========================
# MODELOS NLP
# =========================

models_dir = data_dir / "models"

classification_model_dir = models_dir / "classification_model"
ner_model_dir = models_dir / "ner_model"


# =========================
# CREACIÓN DE CARPETAS
# =========================

# Configuración
domain_config_dir.mkdir(parents=True, exist_ok=True)
ner_config_dir.mkdir(parents=True, exist_ok=True)
asr_config_dir.mkdir(parents=True, exist_ok=True)

# Datasets
ner_dataset_dir.mkdir(parents=True, exist_ok=True)

# Salida NLP
classification_dir.mkdir(parents=True, exist_ok=True)
ner_dir.mkdir(parents=True, exist_ok=True)
final_output_dir.mkdir(parents=True, exist_ok=True)

# Modelos
classification_model_dir.mkdir(parents=True, exist_ok=True)
ner_model_dir.mkdir(parents=True, exist_ok=True)


# =========================
# VERIFICACIÓN
# =========================

print("Ruta raíz:", project_root)

print("\n--- Configuración ---")
print("Dominio:", domain_terms_path)
print("NER:", ner_entities_path)

print("\n--- Datasets ---")
print("NER dataset:", ner_dataset_path)

print("\n--- Input NLP ---")
print("Transcripciones procesadas:", pred_asr_dir)
print("Ground truth:", ground_truth_dir)

print("\n--- Output NLP ---")
print("Clasificación:", classification_dir)
print("NER:", ner_dir)
print("Salida final:", final_output_dir)

print("\n--- Modelos NLP ---")
print("Modelo clasificación:", classification_model_dir)
print("Modelo NER:", ner_model_dir)

## 3. Preparación del dataset para NLP

### 3.1 Carga de transcripciones

In [ ]:
# -------- ASR --------

# Carga de archivos JSON del ASR (pipeline final)
json_paths = sorted(list(pred_asr_dir.glob("*.json")))

dataset_asr = []

for path in json_paths:
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        dataset_asr.append({
            "audio_id": data.get("audio_id", path.stem),
            "text": data.get("text", "")
        })
        
    except Exception as e:
        dataset_asr.append({
            "audio_id": path.stem,
            "text": None,
            "error": str(e)
        })

# Conversión a DataFrame
df_asr = pd.DataFrame(dataset_asr)

print(f"ASR cargado: {len(df_asr)} registros")
df_asr.head()


# -------- GROUND TRUTH --------

# Carga de anotaciones manuales
gt_path = ground_truth_dir / "ground_truth.csv"

df_gt = pd.read_csv(gt_path)

print(f"Ground truth cargado: {len(df_gt)} registros")

# Estandarización de columnas para NLP
df_gt = df_gt.rename(columns={
    "transcripcion": "text",
    "clase": "label"
})

df_gt.head()

### 3.2 Normalización de texto para PLN

Se aplica una normalización ligera del texto con el objetivo de reducir la variabilidad superficial asociada a aspectos como la capitalización, la codificación Unicode y la presencia de espacios redundantes. Este proceso permite homogeneizar la representación del texto sin alterar su contenido semántico, preservando así la información lingüística relevante para las tareas de clasificación y reconocimiento de entidades.

### 3.2.1 Implementación de la normalización de texto

In [ ]:
def normalize_for_nlp(text):
    
    if text is None:
        return ""
    
    # minúsculas
    text = text.lower()
    
    # normalización unicode (mantiene acentos correctamente)
    text = unicodedata.normalize("NFKC", text)
    
    # eliminación de espacios duplicados
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

### 3.2.1 Función de normalización y aplicación al *dataset*

In [ ]:
df_asr["text_clean"] = df_asr["text"].apply(normalize_for_nlp)
df_gt["text_clean"] = df_gt["text"].apply(normalize_for_nlp)

### 3.2 Validación de datos

#### 3.2.1 Validación del dataset ASR

In [ ]:
# Texto válido: no vacío y tipo string
df_asr["text_valid"] = df_asr["text_clean"].apply(
    lambda x: isinstance(x, str) and len(x.strip()) > 0
)

# Filtrado
valid_asr = df_asr[df_asr["text_valid"]].copy()
invalid_asr = df_asr[~df_asr["text_valid"]].copy()

print(f"ASR → válidos: {len(valid_asr)}, inválidos: {len(invalid_asr)}")

# Mostrar ejemplos inválidos si existen
if len(invalid_asr) > 0:
    print("\nEjemplos inválidos ASR:")
    display(invalid_asr.head())

#### 3.2.2 Validación del *dataset Ground Truth*

In [ ]:
# Texto válido
df_gt["text_valid"] = df_gt["text_clean"].apply(
    lambda x: isinstance(x, str) and len(x.strip()) > 0
)

# Label válido (no nulo)
df_gt["label_valid"] = df_gt["label"].notnull()

# Filtrado
valid_gt = df_gt[(df_gt["text_valid"]) & (df_gt["label_valid"])].copy()
invalid_gt = df_gt[~((df_gt["text_valid"]) & (df_gt["label_valid"]))].copy()

print(f"GT → válidos: {len(valid_gt)}, inválidos: {len(invalid_gt)}")

# Mostrar ejemplos inválidos si existen
if len(invalid_gt) > 0:
    print("\nEjemplos inválidos GT:")
    display(invalid_gt.head())

### 3.3 Análisis inicial del *dataset Ground Truth*

In [ ]:
print("Distribución de clases:")
print(valid_gt["label"].value_counts())

print("\nDistribución relativa (%):")
print((valid_gt["label"].value_counts(normalize=True) * 100).round(2))

## 4. Clasificación de mensajes

### 4.1 Definición de clases

meter algo aqui

### 4.2 Preparación del dataset

In [ ]:
# Mantenemos las columnas necesarias para el modelo: identificador, texto limpio y clase.
df_model = valid_gt[["audio_id", "text_clean", "label"]].copy()

# Verificación del tamaño del dataset resultante
print(f"Dataset: {len(df_model)} registros")

# Inspección inicial de los datos
df_model.head()

### 4.3 Codificación de etiquetas

In [ ]:
# Codificación de etiquetas a valores numéricos para el modelo
label_encoder = LabelEncoder()

df_model["label_encoded"] = label_encoder.fit_transform(df_model["label"])

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

print(label_mapping)

### 4.4 División del *dataset* en entrenamiento y test

In [ ]:
# División del dataset en entrenamiento y test manteniendo la distribución de clases
X = df_model["text_clean"]
y = df_model["label_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # Mantener la proporción de clases en ambos conjuntos
)

# Adaptación para modelos basados en transformers (BETO)
X_train_texts = list(X_train)
X_test_texts = list(X_test)

y_train_labels = y_train.reset_index(drop=True)
y_test_labels = y_test.reset_index(drop=True)

### 4.5 Modelado

En este apartado se aborda el modelado del problema de clasificación de mensajes a partir del texto transcrito. Para ello, se plantean dos enfoques diferenciados:
- Un modelo *baseline* basado en representaciones clásicas del texto
- Un modelo basado en transformers mediante el uso de BETO.

El modelo *baseline* se construye utilizando una representación TF-IDF de los textos junto con un clasificador de regresión logística. Esta elección responde a su eficacia contrastada en tareas de clasificación de texto, especialmente en escenarios con conjuntos de datos reducidos, donde ofrece un buen equilibrio entre rendimiento, estabilidad y coste computacional. Además, permite establecer una referencia objetiva sobre la que evaluar posteriormente el impacto del uso de modelos más complejos.

A partir de esta base, se incorpora un segundo enfoque basado en BETO, que permite capturar información contextual más rica del lenguaje. La comparación entre ambos enfoques permitirá analizar si el incremento de complejidad se traduce en una mejora real del rendimiento en el contexto del problema planteado.

#### 4.5.1 Modelo baseline (TF-IDF + *Logistic Regression*)

In [ ]:
# Vectorización TF-IDF y entrenamiento del modelo baseline
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model_baseline = LogisticRegression(max_iter=1000)
model_baseline.fit(X_train_tfidf, y_train)

#### 4.5.2 Modelo basado en BETO (*fine-tuning*)

##### 4.5.2.1 Carga del modelo y tokenizer

In [ ]:
# Carga de tokenizer y modelo BETO para clasificación
model_name = "dccuchile/bert-base-spanish-wwm-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model_beto = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

##### 4.5.2.2 Preparación de los datos (tokenización)

In [ ]:
# Tokenización de los textos para entrada al modelo
train_encodings = tokenizer(
    X_train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    X_test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

##### 4.5.2.3 Construcción del *dataset*

In [ ]:
# Definición de dataset compatible con PyTorch
class TextDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = TextDataset(train_encodings, y_train_labels)
test_dataset = TextDataset(test_encodings, y_test_labels)

##### 4.5.2.4 Configuración del entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_dir="./logs",
    save_strategy="no"
)

##### 4.5.2.5 Entrenamiento y *Fine-tuning* del modelo BETO

In [ ]:
# Inicialización y entrenamiento del modelo
trainer = Trainer(
    model=model_beto,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

### 4.6 Inferencia

#### 4.6.1 Inferencia con modelo *baseline*

In [ ]:
# Predicción sobre el conjunto de test
y_pred = model_baseline.predict(X_test_tfidf)

#### 4.6.2 Inferencia con modelo BETO

In [ ]:
# Predicción sobre el conjunto de test utilizando el modelo BETO entrenado
predictions = trainer.predict(test_dataset)

y_pred_beto = predictions.predictions.argmax(axis=1)

### 4.7 Evaluación

#### 4.7.1 Evaluación del modelo *baseline*

In [ ]:
# Reporte como DataFrame (solo métricas por clase)
report_baseline = classification_report(
    y_test, y_pred,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0
)

df_metrics = pd.DataFrame(report_baseline).transpose().iloc[:-3, :-1]
accuracy = report_baseline["accuracy"]

# Visualización tabular
display(df_metrics)
print(f"\nAccuracy global: {accuracy:.2f}")

# Matriz de confusión (baseline - azul)
cm_baseline = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm_baseline,
            annot=True,
            fmt="d",
            cmap="Blues",
            annot_kws={"size": 14},
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title("Baseline - Matriz de confusión")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

#### 4.7.2 Evaluación del modelo BETO

In [ ]:
# Evaluación visual del modelo BETO
report_beto = classification_report(
    y_test, y_pred_beto,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0
)

df_metrics_beto = pd.DataFrame(report_beto).transpose().iloc[:-3, :-1]
accuracy_beto = report_beto["accuracy"]

# Visualización tabular
display(df_metrics_beto)
print(f"Accuracy global: {accuracy_beto:.2f}")

# Matriz de confusión (BETO - verde)
cm_beto = confusion_matrix(y_test, y_pred_beto)

plt.figure(figsize=(5,4))
sns.heatmap(cm_beto,
            annot=True,
            fmt="d",
            cmap="Greens",
            annot_kws={"size": 14},
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title("BETO - Matriz de confusión")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

#### 4.7.3 Comparación de modelos

PONER ESTO BIEN

En este apartado se comparan los resultados obtenidos por el modelo baseline y el modelo basado en BETO, utilizando las métricas de evaluación previamente calculadas sobre el conjunto de test.

El modelo baseline, basado en TF-IDF y regresión logística, proporciona una referencia inicial del rendimiento del sistema utilizando representaciones clásicas del texto. Por su parte, el modelo BETO permite incorporar información contextual más rica mediante el uso de representaciones profundas del lenguaje.

La comparación se centra en métricas como la exactitud global y el F1-score por clase, así como en el análisis de la matriz de confusión, lo que permite identificar posibles errores sistemáticos en la clasificación.

En esta fase preliminar, los resultados deben interpretarse con cautela debido al tamaño reducido del conjunto de datos, lo que limita la capacidad de generalización de ambos modelos. No obstante, este análisis permite establecer una base metodológica sólida para evaluar el impacto del uso de modelos más complejos cuando se disponga de un volumen de datos mayor y más equilibrado.

##### 4.7.3.1 Comparación de F1 por clase

In [ ]:
# Clases del problema
classes = list(label_encoder.classes_)

# Extracción de métricas F1 desde los reportes
f1_baseline = [report_baseline[c]["f1-score"] for c in classes]
f1_beto = [report_beto[c]["f1-score"] for c in classes]

# Posiciones en el eje X
x = np.arange(len(classes))
width = 0.35
gap = 0.05 # separación adicional entre barras

# Se utilizan tonos derivados de las paletas de seaborn ("Blues" y "Greens")
color_baseline = sns.color_palette("Blues")[4]
color_beto = sns.color_palette("Greens")[4]

# Construcción del gráfico de barras
plt.figure(figsize=(7,4))

plt.bar(x - width/2 - gap/2, f1_baseline, width, label="Baseline", color=color_baseline)
plt.bar(x + width/2 + gap/2, f1_beto, width, label="BETO", color=color_beto)

# Configuración de ejes y etiquetas
plt.xticks(x, classes)
plt.ylabel("F1-score")
plt.title("Comparación de F1-score por clase")
plt.ylim(0,1)

# Leyenda para identificar modelos
plt.legend()

plt.show()

Se observa que ambos modelos presentan valores similares de F1-score en las clases analizadas, sin diferencias significativas en el conjunto de prueba. La exactitud global también muestra un comportamiento equivalente entre ambos enfoques.

##### 4.7.3.2 Comparación de accuracy

In [ ]:
# Extracción de métricas
accuracy_baseline = report_baseline["accuracy"]
accuracy_beto = report_beto["accuracy"]

# Se utilizan tonos derivados de las paletas de seaborn ("Blues" y "Greens")
color_baseline = sns.color_palette("Blues")[4]
color_beto = sns.color_palette("Greens")[4]

# Construcción del gráfico
plt.figure(figsize=(4,3))
plt.bar(
    ["Baseline", "BETO"],
    [accuracy_baseline, accuracy_beto],
    color=[color_baseline, color_beto]
)

# Configuración de ejes
plt.ylim(0,1)
plt.ylabel("Accuracy")
plt.title("Comparación de exactitud")

plt.show()

##### 4.7.3.3 Diferencia de F1

In [ ]:
f1_diff = [b - a for a, b in zip(f1_baseline, f1_beto)]

# Se utilizan tonos derivados de las paletas de seaborn ("Blues" y "Greens")
color_baseline = sns.color_palette("Blues")[4]
color_beto = sns.color_palette("Greens")[4]

# Se utilizan tonos derivados de las paletas de seaborn ("Blues" y "Greens")
colors = [color_beto if v >= 0 else color_baseline for v in f1_diff]

plt.figure(figsize=(7,4))

bars = plt.bar(classes, f1_diff, color=colors)

# Línea base (sin mejora)
plt.axhline(0, color="black", linewidth=1, linestyle="--")

# Etiquetas de valores encima de cada barra
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + (0.02 if height >= 0 else -0.05),
        f"{height:.2f}",
        ha='center'
    )

# Configuración
plt.ylabel("Δ F1-score")
plt.title("Variación del F1-score: BETO vs Baseline")

plt.show()

### 4.8 Análisis final y selección del modelo óptimo

AQUI COMPLETAR CON LAS CONCLUSIONES CON LOS DATOS

Dado el tamaño reducido del dataset disponible en esta fase inicial, ambos modelos presentan un rendimiento similar, sin observarse diferencias significativas en las métricas de evaluación. Este comportamiento indica que el problema se encuentra limitado por la disponibilidad de datos, lo que impide explotar el potencial de modelos más complejos.

En este contexto, el modelo baseline basado en regresión logística se posiciona como una solución adecuada, al ofrecer un rendimiento equivalente con una menor complejidad computacional y mayor interpretabilidad.

Por otro lado, el modelo BETO, basado en arquitecturas de tipo transformer, requiere un mayor volumen de datos para capturar patrones semánticos complejos y generalizar correctamente. Su uso en esta fase se justifica como validación del pipeline y preparación para escenarios futuros.

En consecuencia, se establece como modelo operativo inicial la regresión logística, manteniendo el modelo BETO como candidato principal para futuras iteraciones del sistema, una vez se disponga de un dataset más representativo.

## 5. Optimización y configuración final del modelo seleccionado

INTRO AQUI

### 5.1 Estrategia de optimización

La optimización del modelo se plantea como un proceso sistemático de evaluación de distintas configuraciones de hiperparámetros durante el *fine-tuning* del modelo seleccionado. Dado que se parte de una arquitectura preentrenada, el objetivo no es modificar la estructura del modelo, sino ajustar su comportamiento mediante parámetros que controlan la dinámica del aprendizaje.

La estrategia adoptada consiste en una búsqueda controlada sobre un conjunto reducido de hiperparámetros clave, seleccionados por su impacto directo en el rendimiento del modelo. En particular, se consideran la tasa de aprendizaje (*learning rate*), el tamaño del lote (*batch size*), el número de épocas (*epochs*) y el parámetro de regularización (*weight decay*). Estos parámetros permiten equilibrar la velocidad de convergencia, la estabilidad del entrenamiento y la capacidad de generalización.

En lugar de emplear técnicas automatizadas de optimización, se opta por un enfoque de exploración manual mediante la evaluación de combinaciones discretas de valores. Esta decisión permite analizar de forma directa el efecto de cada hiperparámetro sobre el rendimiento del modelo, facilitando la interpretación de los resultados y manteniendo el control sobre el proceso experimental.

Para garantizar la validez de la comparación, todas las configuraciones se evalúan bajo las mismas condiciones, utilizando la misma partición de datos y las mismas métricas definidas previamente. Además, cada experimento parte de una inicialización limpia del modelo, evitando la acumulación de aprendizaje entre ejecuciones y asegurando la independencia de los resultados.

### 5.2 Definición del espacio de búsqueda

In [ ]:
# Definimos los hiperparámetros a optimizar en el proceso de fine-tuning

learning_rates = [2e-5, 3e-5, 5e-5]   # Tasa de aprendizaje
batch_sizes = [4, 8]                  # Tamaño de lote
epochs_list = [2, 3, 4]               # Número de épocas
weight_decays = [0.0, 0.01, 0.1]      # Regularización L2

### 5.3 Preparación del conjunto de datos para la optimización

Con el objetivo de garantizar la independencia entre los experimentos realizados en los apartados anteriores y el proceso de optimización, se reconstruye la partición del dataset utilizando la misma semilla aleatoria, asegurando la reproducibilidad de los resultados.

#### 5.3.1 División del dataset en entrenamiento, validación y test

In [ ]:
# División inicial (igual que en el apartado 4)
X = df_model["text_clean"]
y = df_model["label_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# División adicional: train → train + validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

#### 5.3.2 Adaptación de los datos al formato requerido por el modelo

In [ ]:
# Adaptación a formato texto
X_train_texts = list(X_train)
X_val_texts   = list(X_val)
X_test_texts  = list(X_test)

# Labels alineadas
y_train_labels = y_train.reset_index(drop=True)
y_val_labels   = y_val.reset_index(drop=True)
y_test_labels  = y_test.reset_index(drop=True)

#### 5.3.3 Tokenización de los textos

In [ ]:
train_encodings = tokenizer(X_train_texts, truncation=True, padding=True, max_length=128)
val_encodings   = tokenizer(X_val_texts,   truncation=True, padding=True, max_length=128)
test_encodings  = tokenizer(X_test_texts,  truncation=True, padding=True, max_length=128)

#### 5.3.4 Construcción de los datasets para entrenamiento

In [ ]:
train_dataset = TextDataset(train_encodings, y_train_labels)
val_dataset   = TextDataset(val_encodings,   y_val_labels)
test_dataset  = TextDataset(test_encodings,  y_test_labels)

### 5.3 Evaluación de combinaciones de hiperparámetros

In [ ]:
# Inicializamos estructura para almacenar resultados
results = []

# Reducimos la verbosidad de los logs para evitar saturación durante la búsqueda de hiperparámetros
logging.set_verbosity_error()

# Filtramos warning específico de PyTorch (MPS)
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*pin_memory.*",
    category=UserWarning
)

# Iteramos sobre todas las combinaciones posibles de hiperparámetros
for lr in learning_rates:
    for batch_size in batch_sizes:
        for epochs in epochs_list:
            for wd in weight_decays:

                print(f"\nEntrenando con: lr={lr}, batch={batch_size}, epochs={epochs}, wd={wd}")

                # Carga del modelo desde cero
                model_beto = AutoModelForSequenceClassification.from_pretrained(
                    model_name,
                    num_labels=len(label_encoder.classes_)
                )

                # Definición de los argumentos de entrenamiento
                training_args = TrainingArguments(
                    output_dir="./results",
                    num_train_epochs=epochs,
                    per_device_train_batch_size=batch_size,
                    per_device_eval_batch_size=batch_size,
                    learning_rate=lr,
                    weight_decay=wd,
                    logging_dir="./logs",
                    save_strategy="no",
                    logging_strategy="no",
                    disable_tqdm=False
                )

                # Definición del Trainer (evaluación sobre validation)
                trainer = Trainer(
                    model=model_beto,
                    args=training_args,
                    train_dataset=train_dataset,
                    eval_dataset=val_dataset
                )

                # Entrenamiento (fine-tuning)
                trainer.train()

                # Evaluación sobre validation
                predictions = trainer.predict(val_dataset)
                y_pred = np.argmax(predictions.predictions, axis=1)

                # Métrica principal: F1 macro
                report = classification_report(
                    y_val_labels,
                    y_pred,
                    output_dict=True,
                    zero_division=0
                )

                f1_macro = report["macro avg"]["f1-score"]

                # Guardamos resultados
                results.append({
                    "learning_rate": lr,
                    "batch_size": batch_size,
                    "epochs": epochs,
                    "weight_decay": wd,
                    "f1_macro": f1_macro
                })

### 5.4 Resultados de la optimización

Se selecciona el ***F1-score macro*** como métrica principal de evaluación, ya que permite valorar el rendimiento del modelo de forma equilibrada entre las distintas clases, evitando sesgos derivados de posibles desbalances en el conjunto de datos. A diferencia de métricas globales como la exactitud, el F1-score integra tanto la precisión como el recall, proporcionando una medida más robusta del comportamiento del modelo en tareas de clasificación.

In [ ]:
df_results = pd.DataFrame(results)

# Ordenamos por métrica principal
df_results_sorted = df_results.sort_values(by="f1_macro", ascending=False)

display(df_results_sorted.head())

### 5.5 Selección de la configuración óptima

* cuál es la mejor combinación
* por qué (F1_macro)

### 5.6 Evaluación final del modelo seleccionado

👉 aquí:

* reentrenas ese modelo
* evalúas en test

#### 5.6.1 Entrenamiento del modelo final

ESTE CODIGO ES PARA CAMBIAR POR LO PRIMERO DE ABAJO EN CASO DE QUE ELIJAMOS LO MISMO, ASI QUE TAMBIEN CAMBIARIAMOS EL COMENTARIO.

In [ ]:
best_config = df_results_sorted.iloc[0]

best_learning_rate = best_config["learning_rate"]
best_batch_size = best_config["batch_size"]
best_epochs = best_config["epochs"]
best_weight_decay = best_config["weight_decay"]

In [ ]:
# Selección manual basada en análisis de resultados
best_learning_rate = 2e-5      # CAMBIAR
best_batch_size = 4            # CAMBIAR
best_epochs = 2                # CAMBIAR
best_weight_decay = 0.0        # CAMBIAR

# Carga del modelo
model_beto_final = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

# Argumentos de entrenamiento
training_args_final = TrainingArguments(
    output_dir="./results_final",
    num_train_epochs=best_epochs,
    per_device_train_batch_size=best_batch_size,
    per_device_eval_batch_size=best_batch_size,
    learning_rate=best_learning_rate,
    weight_decay=best_weight_decay,
    save_strategy="no",
    logging_strategy="no",
    disable_tqdm=False
)

# Trainer
trainer_final = Trainer(
    model=model_beto_final,
    args=training_args_final,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Entrenamiento
trainer_final.train()

#### 5.6.2 Evaluación final en el conjunto de test

In [ ]:
# Predicciones en test
predictions_test = trainer_final.predict(test_dataset)
y_pred_test = np.argmax(predictions_test.predictions, axis=1)

# Métricas
report_test = classification_report(
    y_test_labels,
    y_pred_test,
    target_names=label_encoder.classes_,
    digits=4
)

print("Resultados finales en test:")
print(report_test)

#### 5.6.3 Visualización de resultados

In [ ]:
# Matriz de confusión
cm_test = confusion_matrix(y_test_labels, y_pred_test)

plt.figure(figsize=(5,4))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.title("Matriz de confusión - Evaluación final (Test)")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

#### 5.6.4 Interpretación de resultados y validación del modelo

CONCLUSIONES AQUI

meter tambien: Consideraciones sobre generalización

👉 aquí ganas puntos:

* riesgo de sobreajuste
* dependencia del tamaño del dataset

### 5.7 Guardado del modelo para integración en *pipeline*

EXPLICACIÓN AQUI

In [ ]:
# Guardado del modelo de clasificación
trainer_final.save_model(classification_model_dir)

# Guardado del tokenizer asociado
tokenizer.save_pretrained(classification_model_dir)

# Guardado del codificador de etiquetas
joblib.dump(label_encoder, classification_model_dir / "label_encoder.pkl")

## CUIDADO AL GUARDAR, OCUPA MUCHO PARA GITHUB

## 6. Extracción de entidades (NER)

### 6.1 Validación de la alineación de entidades (*offsets*)

Antes de proceder al entrenamiento del modelo, se realiza una validación del dataset anotado con el objetivo de comprobar la correcta alineación de las entidades definidas mediante offsets.

Este proceso permite detectar errores en los índices de inicio y fin de cada entidad, los cuales deben coincidir exactamente con los límites de los *tokens* generados por spaCy. En caso contrario, dichas entidades serán ignoradas durante el entrenamiento, afectando negativamente al rendimiento del modelo.

Para ello, se emplea la función `offsets_to_biluo_tags`, que permite verificar la correspondencia entre las anotaciones y la tokenización del texto.

<div style="border-left: 5px solid #ff4d4d; padding: 10px; background-color: #2b2b2b; color: #f0f0f0;">
<strong style="color: red">⚠️ Advertencia:</strong> Se recomienda ejecutar este bloque tras cualquier modificación del *dataset* antes de proceder al entrenamiento del modelo.
</div>

In [ ]:
# Carga del archivo de entidades NER
with open(ner_dataset_path, "r", encoding="utf-8") as f:
    ner_data_raw = json.load(f)

errores = []

for i, item in enumerate(ner_data_raw):
    text = item["text"]
    entities = [(ent["start"], ent["end"], ent["label"]) for ent in item["entities"]]

    doc = nlp.make_doc(text)
    tags = offsets_to_biluo_tags(doc, entities)

    if "-" in tags:
        errores.append({
            "index": i,
            "text": text,
            "entities": entities,
            "tags": tags
        })

# Resultado
print(f"Errores encontrados: {len(errores)}")

for e in errores:
    print("\n---")
    print("Index:", e["index"])
    print("Texto:", e["text"])
    print("Entidades:", e["entities"])
    print("Tags:", e["tags"])

### 6.2 Definición de entidades

In [ ]:
# Carga del archivo de entidades NER
with open(ner_entities_path, "r", encoding="utf-8") as f:
    ner_config = json.load(f)

# Lista de entidades definidas
entities = ner_config["entities"]

# Verificación
print("Entidades definidas para NER:")
print(entities)

### 6.3 Preparación del dataset

In [ ]:
# Ruta correcta
ner_dataset_path = data_dir / "datasets" / "ner" / "ner_dataset.json"

# Carga del dataset
with open(ner_dataset_path, "r", encoding="utf-8") as f:
    ner_data_raw = json.load(f)

# Conversión a formato spaCy
train_data = []

for item in ner_data_raw:
    text = item["text"]
    entities = [(ent["start"], ent["end"], ent["label"]) for ent in item["entities"]]
    train_data.append((text, {"entities": entities}))

print("Número de ejemplos:", len(train_data))

### 6.4 Implementación del modelo NER

In [ ]:
# Carga modelo base en español
nlp = spacy.load("es_core_news_lg")

### 6.5 Generación de conjuntos *train*/*validation* para evaluación del modelo NER

In [ ]:
# Se divide el dataset en dos subconjuntos (80% entrenamiento, 20% validación)
train_split, val_split = train_test_split(
    train_data,
    test_size=0.2,       
    random_state=42      
)

# Verificación del tamaño de los conjuntos
print("Train:", len(train_split))
print("Validation:", len(val_split))

### 6.6 *Fine-tuning* del modelo NER basado en spaCy

In [ ]:
# Semilla para reproducibilidad
random.seed(42)
np.random.seed(42)
spacy.util.fix_random_seed(42)

# Configuraciones de entrenamiento (grid)
results = []

configs = [
    {"n_iter": n, "drop": d}
    for n in [10, 20, 30]
    for d in [0.2, 0.3, 0.4]
]

# Entrenamiento y evaluación de configuraciones
for config in configs:

    print("\nEntrenando config:", config)

    # Carga del modelo base
    nlp = spacy.load("es_core_news_md")
    ner = nlp.get_pipe("ner")

    # Añadir etiquetas del conjunto de entrenamiento
    for _, annotations in train_split:
        for ent in annotations["entities"]:
            ner.add_label(ent[2])

    optimizer = nlp.resume_training()

    # Inicialización del early stopping
    best_loss = float("inf")
    patience = 3
    no_improve = 0
    epochs_run = 0

    # Bucle de entrenamiento
    for epoch in range(config["n_iter"]):

        random.shuffle(train_split)
        losses = {}

        for text, annotations in train_split:
            doc = nlp.make_doc(text)
            example = Example.from_dict(doc, annotations)

            nlp.update(
                [example],
                drop=config["drop"],
                losses=losses
            )

        current_loss = losses.get("ner", 0.0)
        epochs_run += 1

        # Early stopping: mejora del modelo
        if current_loss < best_loss:
            best_loss = current_loss
            no_improve = 0
        else:
            # Early stopping: no mejora acumulada
            no_improve += 1

        # Early stopping: condición de parada
        if no_improve >= patience:
            print("Early stopping activado")
            break

    # Evaluación en conjunto de validación
    scorer = Scorer()
    examples = []

    for text, annotations in val_split:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)

        pred_doc = nlp(text)
        example = Example(pred_doc, example.reference)

        examples.append(example)

    scores = scorer.score(examples)

    # Registro de resultados
    results.append({
        "n_iter": config["n_iter"],
        "drop": config["drop"],
        "epochs_run": epochs_run,
        "loss": best_loss,
        "precision": scores["ents_p"],
        "recall": scores["ents_r"],
        "f1": scores["ents_f"]
    })

    print("F1:", scores["ents_f"])

### 6.7 Análisis de resultados del *fine-tuning*

In [ ]:
# Conversión de los resultados a DataFrame para facilitar su análisis
df_results = pd.DataFrame(results)

# Ordenación de las configuraciones evaluadas:
# - F1 como métrica principal
# - Precision y recall como criterios secundarios
# - Loss como criterio de desempate (menor es mejor)
df_results_sorted = df_results.sort_values(
    by=["f1", "precision", "recall", "loss"],
    ascending=[False, False, False, True]
)

# Visualización de resultados ordenados
display(df_results_sorted)

aqui comentamos y elegimos la mejor

✔️ F1 → métrica principal

✔️ precision → calidad detección

✔️ recall → cobertura

✔️ loss → solo referencia

In [ ]:
plt.scatter(df_results["drop"], df_results["f1"])
plt.xlabel("Dropout")
plt.ylabel("F1")
plt.title("Impacto del dropout en el rendimiento")
plt.show()

In [ ]:
plt.plot(df_results_sorted["f1"].values)

### 6.8 Entrenamiento final del modelo NER tras selección de configuración óptima

In [ ]:
# Semilla para reproducibilidad
random.seed(42)
np.random.seed(42)
spacy.util.fix_random_seed(42)

# Selección de la mejor configuración
best_config = df_results_sorted.iloc[0].to_dict()
print("Mejor configuración:", best_config)

# Carga del modelo base
nlp = spacy.load("es_core_news_md")
ner = nlp.get_pipe("ner")

# Añadir etiquetas del dataset completo
for _, annotations in train_data:
    for ent in annotations["entities"]:
        ner.add_label(ent[2])

optimizer = nlp.resume_training()

# Configuración de early stopping
best_loss = float("inf")
patience = 3
no_improve = 0

# Entrenamiento final del modelo
for epoch in range(int(best_config["n_iter"])):

    random.shuffle(train_data)
    losses = {}

    for text, annotations in train_data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)

        nlp.update(
            [example],
            drop=best_config["drop"],
            losses=losses
        )

    current_loss = losses.get("ner", 0.0)

    print(f"Epoch {epoch+1} - Loss: {current_loss}")

    if current_loss < best_loss:
        best_loss = current_loss
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= patience:
        print("Early stopping activado")
        break

### 6.9 Análisis cualitativo del modelo NER

aqui poner una frase mas complicada

In [ ]:
def extract_entities(text):
    # Normalización ligera del texto antes de aplicar el modelo
    text = normalize_for_nlp(text)
    
    # Procesamiento del texto mediante el modelo NER entrenado
    doc = nlp(text)
    
    # Extracción de entidades detectadas (texto y etiqueta)
    return [(ent.text, ent.label_) for ent in doc.ents]


# Conjunto de ejemplos representativos para análisis cualitativo del modelo
textos = [
    "Ayer al pasarme por el cafetal, me encontré roya en el cafeto y algunas hojas estaban amarillas",
    "La semana pasada estuvimos revisando la parcela y vimos broca en varios granos, pero no aplicamos nada todavía",
    "Después de las lluvias fuertes, apareció monilia en el cacao y se está dañando la mazorca",
    "Fui al lote 3 y noté que el café está en floración pero también hay presencia de ojo de gallo en algunas hojas"
]

# Evaluación del comportamiento del modelo sobre distintos escenarios
for texto in textos:
    print("\nTexto:", texto)
    
    # Aplicación del modelo NER sobre cada ejemplo
    entidades = extract_entities(texto)
    
    # Visualización de entidades detectadas
    print("Entidades detectadas:", entidades)

AQUI COMENTAR QUE HAY COSAS QUE NO DETECTA Y PORQUE Y ASI JUSTIFICAMOS EL USO DEL MODELO HIBRIDO INTEGRANDO DICCIONARIOS.

El análisis cualitativo del modelo permite observar que, si bien es capaz de identificar correctamente ciertas entidades frecuentes como acciones o cultivos, presenta limitaciones en la detección de entidades específicas del dominio agrícola, como plagas o enfermedades (por ejemplo, "roya", "broca" o "monilia").

Este comportamiento es coherente con los resultados obtenidos en la evaluación cuantitativa, donde se observa un valor elevado de precisión pero un recall reducido, lo que indica una detección conservadora de entidades.

Estas limitaciones justifican la necesidad de incorporar mecanismos adicionales que permitan reforzar la detección de términos relevantes del dominio.

### 6.10 Refuerzo mediante conocimiento de dominio (enfoque híbrido)

Explicacion completa: Para evitar la generación de duplicados y posibles inconsistencias en las entidades detectadas, se incorpora un mecanismo de consolidación basado en la priorización de las entidades identificadas por el modelo NER frente a las obtenidas mediante el diccionario de dominio. De este modo, el diccionario actúa únicamente como mecanismo de refuerzo, añadiendo nuevas entidades cuando estas no han sido previamente detectadas por el modelo.

In [ ]:
# Carga del diccionario de dominio
with open(domain_terms_path, "r", encoding="utf-8") as f:
    domain_terms = json.load(f)


# Función de extracción híbrida: modelo NER + diccionario de dominio
def extract_entities_with_domain(text):
    
    # Normalización del texto (clave)
    text = normalize_for_nlp(text)

    # Procesamiento del texto con el modelo NER
    doc = nlp(text)

    # Diccionario para evitar duplicados
    entities_dict = {}

    # Entidades detectadas por el modelo (prioridad)
    for ent in doc.ents:
        key = ent.text.lower()
        entities_dict[key] = ent.label_

    # Texto ya normalizado
    text_lower = text.lower()

    # Refuerzo mediante diccionario de dominio
    for categoria, conceptos in domain_terms.items():
        for concepto, variantes in conceptos.items():
            for variante in variantes:

                # Coincidencia exacta de palabra
                patron = r"\b" + re.escape(variante.lower()) + r"\b"

                if re.search(patron, text_lower):
                    key = variante.lower()

                    # Solo añadir si no existe ya
                    if key not in entities_dict:
                        entities_dict[key] = categoria

    # Convertir a lista
    return [(k, v) for k, v in entities_dict.items()]


# Conjunto de ejemplos representativos para análisis cualitativo del modelo
textos = [
    "Ayer al pasarme por el cafetal, me encontré roya en el cafeto y algunas hojas estaban amarillas",
    "La semana pasada estuvimos revisando la parcela y vimos broca en varios granos, pero no aplicamos nada todavía",
    "Después de las lluvias fuertes, apareció monilia en el cacao y se está dañando la mazorca",
    "Fui al lote 3 y noté que el café está en floración pero también hay presencia de ojo de gallo en algunas hojas"
]

# Evaluación del comportamiento del modelo sobre distintos escenarios
for texto in textos:
    print("\nTexto:", texto)
    
    # Resultado del modelo NER
    entidades = extract_entities(texto)
    print("NER:", entidades)
    
    # Resultado del enfoque híbrido
    print("Híbrido:", extract_entities_with_domain(texto))

### 6.11 Guardado del modelo para integración en *pipeline*

In [ ]:
# Guardado del modelo NER
nlp.to_disk(ner_model_dir)

# Verificación de guardado
print("Modelo NER guardado en:", ner_model_dir)

## 7. Implementación y ejecución del *pipeline* de clasificación y NER

El presente apartado describe la implementación del *pipeline* completo de procesamiento del lenguaje natural, integrando las etapas de clasificación de mensajes y extracción de entidades. El objetivo es transformar el texto transcrito generado por el sistema ASR en una representación estructurada preliminar, identificando tanto la categoría del mensaje como las entidades relevantes presentes en el mismo.

Al igual que en el módulo de transcripción, se adopta un enfoque modular basado en funciones independientes, lo que permite mantener una separación clara de responsabilidades y facilita tanto la validación del sistema como su integración dentro del *pipeline* global.

### 7.1 Inicialización del entorno y gestión de rutas

In [ ]:
# Configuración de rutas del pipeline de clasificación y NER
# Permite localizar automáticamente la estructura del proyecto
# y definir las carpetas de entrada (ASR) y salida (NLP)

def configure_paths(
    base_data_dir: str = "data",
    input_folder: str = "transcriptions/asr_output",
    output_folder: str = "structured_data"
) -> dict:

    project_root = Path.cwd()

    # Detectar raíz del proyecto
    while not (project_root / base_data_dir).exists():
        project_root = project_root.parent

    data_dir = project_root / base_data_dir

    # Carpeta de entrada (salida del ASR)
    input_text_dir = data_dir / input_folder

    # Carpeta base de salida NLP
    structured_data_dir = data_dir / output_folder

    # Subcarpetas específicas
    classification_dir = structured_data_dir / "classification"
    ner_dir = structured_data_dir / "ner"

    # Crear carpetas si no existen
    classification_dir.mkdir(parents=True, exist_ok=True)
    ner_dir.mkdir(parents=True, exist_ok=True)

    return {
        "project_root": project_root,
        "input_text_dir": input_text_dir,
        "classification_dir": classification_dir,
        "ner_dir": ner_dir
    }

### 7.2 Implementación de la función de normalización de texto

In [ ]:
def normalize_for_nlp(text: str) -> str:
    
    if text is None:
        return ""
    
    # Conversión a minúsculas
    text = text.lower()
    
    # Normalización Unicode
    text = unicodedata.normalize("NFKC", text)
    
    # Eliminación de espacios duplicados
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

### 7.3 Implementación de funciones del *pipeline* de clasificación y extracción de entidades

INTRO AQUI

#### 7.2.1 Carga del *dataset* de transcripciones

In [ ]:
# Carga las transcripciones generadas por el ASR
# Devuelve una lista de diccionarios con audio_id y texto

def load_text_dataset(input_text_dir: Path) -> list[dict]:

    json_files = list(input_text_dir.glob("*.json"))

    if len(json_files) == 0:
        raise FileNotFoundError("No se han encontrado transcripciones en el directorio de entrada")

    dataset = []

    for f in json_files:
        with open(f, "r", encoding="utf-8") as file:
            data = json.load(file)

            text_raw = data["text"]

            # Normalización en carga
            text_clean = normalize_for_nlp(text_raw)

            dataset.append({
                "audio_id": data["audio_id"],
                "text": text_raw,
                "text_clean": text_clean
            })

    return dataset

#### 7.2.2 Inicialización de modelos de clasificación y NER

In [ ]:
# Carga los modelos previamente entrenados
# Clasificación (BETO)
# NER (modelo spaCy fine-tuned)

def load_nlp_models(
    classification_model_path: Path,
    ner_model_path: Path
):

    classification_model = joblib.load(classification_model_path)

    nlp = spacy.load(ner_model_path)

    return classification_model, nlp

#### 7.2.3 Clasificación de mensajes

In [ ]:
# Aplica el modelo de clasificación a un texto
# Devuelve la etiqueta predicha

def classify_text(text: str, model) -> str:

    prediction = model.predict([text])[0]

    return prediction

#### 7.2.4 Extracción de entidades (NER)

In [ ]:
# Aplica el modelo NER sobre el texto
# Devuelve lista de entidades detectadas

def extract_entities(text: str, nlp) -> list:

    doc = nlp(text)

    return [(ent.text, ent.label_) for ent in doc.ents]

#### 7.2.5 Refuerzo mediante conocimiento de dominio

In [ ]:
# Refuerza la detección de entidades mediante diccionario de dominio
# Evita duplicados y prioriza las entidades detectadas por el modelo NER

def extract_entities_with_domain(text: str, nlp, domain_terms: dict) -> list:

    doc = nlp(text)

    # Diccionario para evitar duplicados
    entities_dict = {}

    # Entidades detectadas por el modelo (prioridad)
    for ent in doc.ents:
        key = ent.text.lower()
        entities_dict[key] = ent.label_

    # Conversión a minúsculas
    text_lower = text.lower()

    # Refuerzo mediante diccionario de dominio
    for categoria, conceptos in domain_terms.items():
        for concepto, variantes in conceptos.items():
            for variante in variantes:

                # Escapamos caracteres especiales y buscamos palabra completa
                patron = r"\b" + re.escape(variante.lower()) + r"\b"

                if re.search(patron, text_lower):

                    key = variante.lower()

                    # Solo añadir si no existe ya
                    if key not in entities_dict:
                        entities_dict[key] = categoria

    # Convertir a lista de tuplas
    return [(k, v) for k, v in entities_dict.items()]

#### 7.2.6 Generación de salida estructurada preliminar

In [ ]:
# Genera estructura final del pipeline NLP (sin normalización)

def process_text(
    item: dict,
    classification_model,
    nlp,
    domain_terms
) -> dict:

    text_clean = item["text_clean"]

    label = classify_text(text_clean, classification_model)

    entities = extract_entities_with_domain(text_clean, nlp, domain_terms)

    return {
        "audio_id": item["audio_id"],
        "text": item["text"],          # raw
        "text_clean": text_clean,      # limpio
        "label": label,
        "entities": entities
    }

### 7.3 Ejecución del *pipeline* de clasificación y NER

In [ ]:
print("Inicializando pipeline NLP...")

# Configuración de rutas
paths = configure_paths()
print("Rutas configuradas correctamente")

# Carga de transcripciones
print("Cargando dataset de transcripciones...")
text_dataset = load_text_dataset(paths["input_text_dir"])
print(f"Registros cargados: {len(text_dataset)}")

# Carga de modelos
print("Cargando modelos NLP...")
classification_model_path = models_dir / "classification_model"
ner_model_path = models_dir / "ner_model"

classification_model, nlp = load_nlp_models(
    classification_model_path,
    ner_model_path
)
print("Modelos cargados correctamente")

# Carga del diccionario de dominio
print("Cargando diccionario de dominio...")
with open(domain_terms_path, "r", encoding="utf-8") as f:
    domain_terms = json.load(f)
print("Diccionario cargado correctamente")

# Procesamiento del dataset
print("Procesando textos...")
results = []

for item in text_dataset:
    processed = process_text(
        item,
        classification_model,
        nlp,
        domain_terms
    )
    results.append(processed)

print(f"Registros procesados: {len(results)}")

# Guardado de resultados
print("Guardando resultados...")

for item in results:

    # Guardado clasificación
    output_classification_path = paths["classification_dir"] / f"{item['audio_id']}.json"

    with open(output_classification_path, "w", encoding="utf-8") as f:
        json.dump({
            "audio_id": item["audio_id"],
            "text": item["text"],
            "text_clean": item["text_clean"],
            "label": item["label"]
        }, f, ensure_ascii=False, indent=4)

    # Guardado NER
    output_ner_path = paths["ner_dir"] / f"{item['audio_id']}.json"

    with open(output_ner_path, "w", encoding="utf-8") as f:
        json.dump({
            "audio_id": item["audio_id"],
            "entities": item["entities"]
        }, f, ensure_ascii=False, indent=4)

print("Resultados guardados correctamente")

print("Pipeline NLP completado correctamente")

### 7.4 Validación del *pipeline* de clasificación y NER

In [ ]:
print("Iniciando validación del pipeline NLP...")

# Verificación de conteo
assert len(results) == len(text_dataset), "Error: número de resultados no coincide"
print("✔ Conteo correcto")

# Verificación de estructura y contenido
audio_ids = set()

for item in results:

    # Campos obligatorios
    assert "audio_id" in item, "Error: falta 'audio_id'"
    assert "text" in item, "Error: falta 'text'"
    assert "text_clean" in item, "Error: falta 'text_clean'"
    assert "label" in item, "Error: falta 'label'"
    assert "entities" in item, "Error: falta 'entities'"

    # Tipos
    assert isinstance(item["text"], str), "Error: 'text' no es string"
    assert isinstance(item["text_clean"], str), "Error: 'text_clean' no es string"
    assert isinstance(item["label"], str), "Error: 'label' no es string"
    assert isinstance(item["entities"], list), "Error: 'entities' no es lista"

    # Contenido
    assert item["text"].strip() != "", f"Texto vacío en {item['audio_id']}"
    assert item["text_clean"].strip() != "", f"Texto limpio vacío en {item['audio_id']}"

    # Unicidad
    assert item["audio_id"] not in audio_ids, f"audio_id duplicado: {item['audio_id']}"
    audio_ids.add(item["audio_id"])

print("✔ Estructura y contenido correctos")
print("✔ Unicidad de audio_id verificada")

# Verificación básica de entidades
for item in results:
    for ent in item["entities"]:
        assert isinstance(ent, tuple), "Entidad mal formada (no es tupla)"
        assert len(ent) == 2, "Entidad mal formada (estructura incorrecta)"

print("✔ Formato de entidades correcto")

# Muestra para inspección manual
print("\n=== Muestra de resultados NLP ===")
sample = pd.DataFrame(results).sample(n=min(5, len(results)), random_state=42)
display(sample)

print("Validación completada correctamente")

## 8. Conclusiones